# Notebook 03 — Data Cleaning

## Objectives
* Remove columns with no analytical or predictive value
* Investigate and document missing values
* Investigate and handle duplicate records
* Investigate outliers and document decisions
* Standardise categorical variables
* Clean Dataset 2 (Marketing KPIs) for ROI analysis
* Save cleaned datasets to inputs/datasets/cleaned/

## Inputs
* `inputs/datasets/raw/digital_marketing_campaign_dataset.csv`
* `inputs/datasets/raw/Marketing.csv`

## Outputs
* `inputs/datasets/cleaned/marketing_campaign_cleaned.csv`
* `inputs/datasets/cleaned/marketing_kpis_cleaned.csv`

## Notes
* This notebook fulfils the CRISP-DM Data Preparation step (criterion 7.2)
* Missing value investigation is documented even when no missings are found
* Outlier decisions are documented with rationale
* All decisions are justified before applying any transformation

In [1]:
import os
import pandas as pd
import numpy as np

os.chdir(os.path.dirname(os.getcwd()))
os.makedirs('inputs/datasets/cleaned', exist_ok=True)
print(f"Working directory: {os.getcwd()}")

Working directory: /workspaces/digital-marketing-conversion-predictor


---
## Dataset 1 — Digital Marketing Campaign

### 1.1 Load and Inspect

In [2]:
df = pd.read_csv('inputs/datasets/raw/digital_marketing_campaign_dataset.csv')
print(f"Original shape: {df.shape}")
df.head()

Original shape: (8000, 20)


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid,1
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid,1
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid,1


### 1.2 Drop Non-Useful Columns

In [3]:
# CustomerID: identifier, no predictive value
# AdvertisingPlatform / AdvertisingTool: confidential, single value only
# ConversionRate: derived metric — would cause data leakage in ML
cols_to_drop = ['CustomerID', 'AdvertisingPlatform', 'AdvertisingTool', 'ConversionRate']
df = df.drop(columns=cols_to_drop)
print(f"Shape after dropping columns: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Shape after dropping columns: (8000, 16)
Remaining columns: ['Age', 'Gender', 'Income', 'CampaignChannel', 'CampaignType', 'AdSpend', 'ClickThroughRate', 'WebsiteVisits', 'PagesPerVisit', 'TimeOnSite', 'SocialShares', 'EmailOpens', 'EmailClicks', 'PreviousPurchases', 'LoyaltyPoints', 'Conversion']


### 1.3 Missing Values Investigation

In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})
print("Missing values report:")
print(missing_report)
print(f"\nTotal missing values: {missing.sum()}")

Missing values report:
                   Missing Count  Missing %
Age                            0        0.0
Gender                         0        0.0
Income                         0        0.0
CampaignChannel                0        0.0
CampaignType                   0        0.0
AdSpend                        0        0.0
ClickThroughRate               0        0.0
WebsiteVisits                  0        0.0
PagesPerVisit                  0        0.0
TimeOnSite                     0        0.0
SocialShares                   0        0.0
EmailOpens                     0        0.0
EmailClicks                    0        0.0
PreviousPurchases              0        0.0
LoyaltyPoints                  0        0.0
Conversion                     0        0.0

Total missing values: 0


**Finding:** No missing values found in any column. No imputation required.
This is documented here to demonstrate compliance with the CRISP-DM
Data Preparation stage (criterion 7.2).

### 1.4 Duplicate Records Investigation

In [5]:
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates found. No action required.")

Duplicate rows found: 0
No duplicates found. No action required.


### 1.5 Outlier Investigation

In [6]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Conversion']

outlier_report = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report.append({
        'Feature': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'Lower Bound': round(lower, 2),
        'Upper Bound': round(upper, 2),
        'N Outliers': n_out,
        'Outlier %': f"{n_out/len(df)*100:.1f}%"
    })

pd.DataFrame(outlier_report)

,Feature,Q1,Q3,Lower Bound,Upper Bound,N Outliers,Outlier %
0,Age,31.00,56.00,-6.50,93.50,0,0.0%
1,Income,51744.50,116815.75,-45862.38,214422.62,0,0.0%
2,AdSpend,2523.22,7407.99,-4803.93,14735.14,0,0.0%
3,ClickThroughRate,0.08,0.23,-0.14,0.45,0,0.0%
4,WebsiteVisits,13.00,37.00,-23.00,73.00,0,0.0%
5,PagesPerVisit,3.30,7.84,-3.50,14.64,0,0.0%
6,TimeOnSite,4.07,11.48,-7.05,22.60,0,0.0%
7,SocialShares,25.00,75.00,-50.00,150.00,0,0.0%
8,EmailOpens,5.00,14.00,-8.50,27.50,0,0.0%
9,EmailClicks,2.00,7.00,-5.50,14.50,0,0.0%


**Outlier Decision:** No outliers detected by IQR method in any feature.
All values fall within reasonable real-world ranges for digital marketing data.
No outlier removal required. We will use a tree-based model (Random Forest)
which is inherently robust to outliers in any case.

### 1.6 Categorical Variables Inspection

In [7]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f"\n{col} ({df[col].nunique()} unique values):")
    print(df[col].value_counts())


Gender (2 unique values):
Gender
Female    4839
Male      3161
Name: count, dtype: int64

CampaignChannel (5 unique values):
CampaignChannel
Referral        1719
PPC             1655
Email           1557
SEO             1550
Social Media    1519
Name: count, dtype: int64

CampaignType (4 unique values):
CampaignType
Conversion       2077
Awareness        1988
Consideration    1988
Retention        1947
Name: count, dtype: int64


### 1.7 Save Cleaned Dataset 1

In [8]:
df.to_csv('inputs/datasets/cleaned/marketing_campaign_cleaned.csv', index=False)
print(f"Saved: inputs/datasets/cleaned/marketing_campaign_cleaned.csv")
print(f"Final shape: {df.shape}")
df.head()

Saved: inputs/datasets/cleaned/marketing_campaign_cleaned.csv
Final shape: (8000, 16)


,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,Conversion
0,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0,2.399017,7.396803,19,6,9,4,688,1
1,69,Male,41760,Email,Retention,3898.668606,0.155725,42,2.917138,5.352549,5,2,7,2,3459,1
2,46,Female,88456,PPC,Awareness,1546.429596,0.277490,2,8.223619,13.794901,0,11,2,8,2337,1
3,32,Female,44085,PPC,Conversion,539.525936,0.137611,47,4.540939,14.688363,89,2,2,0,2463,1
4,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0,2.046847,13.993370,6,6,6,8,4345,1


---
## Dataset 2 — Marketing KPIs

### 2.1 Load and Inspect

In [9]:
df2 = pd.read_csv('inputs/datasets/raw/Marketing.csv')
print(f"Shape: {df2.shape}")
df2.head()

Shape: (308, 11)


,id,c_date,campaign_name,category,campaign_id,impressions,mark_spent,clicks,leads,orders,revenue
0,1,2021-02-01,facebook_tier1,social,349043,148263,7307.37,1210,13,1,4981.0
1,2,2021-02-01,facebOOK_tier2,social,348934,220688,16300.20,1640,48,3,14962.0
2,3,2021-02-01,google_hot,search,89459845,22850,5221.60,457,9,1,7981.0
3,4,2021-02-01,google_wide,search,127823,147038,6037.00,1196,24,1,2114.0
4,5,2021-02-01,youtube_blogger,influencer,10934,225800,29962.20,2258,49,10,84490.0


### 2.2 Standardise Text and Parse Dates

In [10]:
# Standardise campaign_name and category to lowercase
df2['campaign_name'] = df2['campaign_name'].str.lower().str.strip()
df2['category'] = df2['category'].str.lower().str.strip()

# Parse date column
df2['c_date'] = pd.to_datetime(df2['c_date'])

# Add month column for trend analysis
df2['month'] = df2['c_date'].dt.to_period('M').astype(str)

# Add ROI column: (revenue - spend) / spend
df2['roi'] = ((df2['revenue'] - df2['mark_spent']) / df2['mark_spent'].replace(0, np.nan)).round(2)

print("Campaign names after standardisation:")
print(df2['campaign_name'].unique())
print(f"\nCategories: {df2['category'].unique()}")
print(f"\nShape after cleaning: {df2.shape}")
df2.head()

Campaign names after standardisation:
['facebook_tier1' 'facebook_tier2' 'google_hot' 'google_wide'
 'youtube_blogger' 'instagram_tier1' 'instagram_tier2'
 'facebook_retargeting' 'facebook_lal' 'instagram_blogger'
 'banner_partner']

Categories: ['social' 'search' 'influencer' 'media']

Shape after cleaning: (308, 13)


/tmp/ipykernel_15632/718546038.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df2['campaign_name'] = df2['campaign_name'].str.lower().str.strip()
/tmp/ipykernel_15632/718546038.py:3: FutureWarning: ChainedAssignmentError: behaviour will

,id,c_date,campaign_name,category,campaign_id,impressions,mark_spent,clicks,leads,orders,revenue,month,roi
0,1,2021-02-01,facebook_tier1,social,349043,148263,7307.37,1210,13,1,4981.0,2021-02,-0.32
1,2,2021-02-01,facebook_tier2,social,348934,220688,16300.20,1640,48,3,14962.0,2021-02,-0.08
2,3,2021-02-01,google_hot,search,89459845,22850,5221.60,457,9,1,7981.0,2021-02,0.53
3,4,2021-02-01,google_wide,search,127823,147038,6037.00,1196,24,1,2114.0,2021-02,-0.65
4,5,2021-02-01,youtube_blogger,influencer,10934,225800,29962.20,2258,49,10,84490.0,2021-02,1.82


### 2.3 Save Cleaned Dataset 2

In [11]:
df2.to_csv('inputs/datasets/cleaned/marketing_kpis_cleaned.csv', index=False)
print(f"Saved: inputs/datasets/cleaned/marketing_kpis_cleaned.csv")
print(f"Final shape: {df2.shape}")

Saved: inputs/datasets/cleaned/marketing_kpis_cleaned.csv
Final shape: (308, 13)


---
## Summary

| Action | Dataset 1 | Dataset 2 |
|---|---|---|
| Columns dropped | 4 (CustomerID, AdvertisingPlatform, AdvertisingTool, ConversionRate) | None |
| Missing values | 0 — no imputation required | 0 |
| Duplicates removed | 0 | 0 |
| Outliers removed | 0 — tree model is robust | N/A |
| Text standardisation | Not needed | campaign_name and category lowercased |
| New columns added | None | month, roi |
| Output shape | (8000, 16) | (308, 13) |

Both cleaned datasets saved to `inputs/datasets/cleaned/`.
Ready for Feature Engineering notebook.